In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path

from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.preprocessing import Binarizer
from sklearn.preprocessing import QuantileTransformer
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

# 01. Importación
---

## 1.1. Importación de datos

In [5]:
# Definir rutas
data_path = Path('../02_datos/01_Originales')
nombre_fichero = 'Leads.csv'

# Importar Leads.csv
leads_file = data_path / nombre_fichero

df = pd.read_csv(leads_file, sep=';', encoding='utf-8', index_col='id')

print(f'✅ Importación completada')
print(f'Shape: {df.shape}')

✅ Importación completada
Shape: (9093, 20)


## 1.2. Seleccionar variables finales

In [9]:
ruta_proyecto = "../02_datos/03_Entrenamiento/"
nombres_variables_finales = ruta_proyecto + '07_df_final.pkl'

pd.read_pickle(nombres_variables_finales).sort_index().columns.to_list()

['tiempo_en_site_total_mms',
 'score_actividad_mms',
 'ult_actividad_SMS Sent',
 'paginas_vistas_visita_mms',
 'visitas_total_mms',
 'score_perfil_mms',
 'ocupacion_Working Professional',
 'ambito_Select',
 'ult_actividad_Chat Conversation',
 'ocupacion_Unemployed',
 'ult_actividad_Page Visited on Website',
 'ult_actividad_Converted to Lead',
 'descarga_lm_No',
 'compra']

In [21]:
variables_finales = ['ambito',
                     'descarga_lm',
                     'ocupacion',
                     'paginas_vistas_visita',
                     'score_actividad',
                     'score_perfil', 
                     'tiempo_en_site_total', 
                     'ult_actividad',
                     'visitas_total']

In [ ]:
# Cambiar tipo de dato

df['visitas_total'] = df['visitas_total'].astype('Int64')

# 02. Calidad de Datos
---

![alt text](<Captura de pantalla 2026-06-15 a las 15.33.10.png>)

### Duplicados

In [14]:
df.drop_duplicates(inplace=True)

### Por EDA

In [16]:
df = df.loc[(df.no_llamar != 'OTROS') & (df.no_enviar_email != 'Yes') & (df.ult_actividad != 'Email Bounced')]

# 03. Divir en X e Y
---

In [24]:
x = df[variables_finales].copy()

In [27]:
target = 'compra'
y = df[target].copy()

# 04. Pipeline
---

## 4.1. Calidad de Datos

In [ ]:
# Duplicados
df.drop_duplicates(inplace=True)

# Identificacion de nulos en categóricas
cat = df.select_dtypes(exclude = 'number').copy()
num = df.select_dtypes(include = 'number').copy()


### Categóricas

In [ ]:
# Imputar por la moda toda las cat
def imputar_moda(variables):
    for variable in variables:
        cat[variable] = cat[variable].fillna(cat[variable].mode()[0])

imputar_moda(cat)

# Imputar por la moda toda las cat


# Atipicos o raras

def agrupar_cat_raras(variable, criterio = 0.05):
    #Calcula las frecuencias
    frecuencias = variable.value_counts(normalize=True)

    #Identifica las que están por debajo 
    raras = frecuencias[frecuencias < criterio].index

    return np.where(variable.isin(raras), 'OTROS', variable)

variables_agrupar_categorias = cat.columns.to_list()
criterio_agrupar = 0.02

for variable in variables_agrupar_categorias:
    cat[variable] = agrupar_cat_raras(cat[variable], criterio=criterio_agrupar)

### Numéricas

In [ ]:
# Imputar mediana

def imputar_mediana (variable):
    if pd.api.types.is_int64_dtype:
        return(variable.fillna(int(variable.median())))
    
    else:
        return variable.fillna(variable.median())

num[cat] = num[cat].apply(imputar_mediana)


# Atípicos (por límites)

def atipicos_des_tip(variable, num_desv_tip = 4):
    # Sacamos los nulos por ahora
    variable = variable.dropna()

    # Calculamos los límites
    media = np.mean(variable)
    sd = np.std(variable)
    umbral = sd * num_desv_tip

    lim_inf = media - umbral
    lim_sup = media + umbral

    # Encontramos los indices de los que están fuera de los límites
    return variable[(variable < lim_inf) | (variable > lim_sup)].index.tolist()


# Función que cuenta el número de atípicos por variable

def conteo_atipicos(df, variable, num_desv_tip=4):
    atipicos = atipicos_des_tip(df[variable], num_desv_tip)
    return (df.loc[atipicos, variable].value_counts())


for variable in num:
    print('\n' + variable + ':\n')
    print(conteo_atipicos(num, variable, num_desv_tip))

# Winsorización manual

df['visitas_total'] = df['visitas_total'].clip(0, 50)
df['paginas_vistas_visita'] = df['paginas_vistas_visita'].clip(0, 20)

# Imputar nulos en score_actividad y score_perfil por ceros
cols_imputar = ['score_actividad', 'score_perfil']

imputado = df[cols_imputar].isnull().any(axis=1).astype(int)

# crear variable usuario_nuevo
df['usuario_nuevo'] = imputado

## 4.2. EDA

## 4.3. Transformación

### Categóricas

In [ ]:
var_ohe = ['origen', 'fuente', 'ult_actividad', 'ambito', 'ocupacion', 'descarga_lm']

### One-Hot Encoding ###

from sklearn.preprocessing import OneHotEncoder

# Instanciamos OHE
ohe = OneHotEncoder(sparse_output = False, handle_unknown = 'ignore')

# Entrenamos y aplicar
cat_ohe = ohe.fit_transform(cat[var_ohe])

# Guardamos como dataframe
cat_ohe = pd.DataFrame(cat_ohe, columns=ohe.get_feature_names_out())

### Numéricas

In [ ]:
var_mms = ['paginas_vistas_vista', 'score_actividad', 'score_perfil', 'visitas_total', 'tiempo_en_site_total']


### Min-Max Scaling ###

from sklearn.preprocessing import MinMaxScaler

# Instanciamos 
mms = MinMaxScaler()

# Entrenamos y aplicar
df_mms = mms.fit_transform(df[var_mms])

# Añadimos sujijos a los nombres

nombres_mms = [variable + '_mms' for variable in var_mms]

# Guardamos en dataframe
df_mms = pd.DataFrame(df_mms, columns=nombres_mms)

# 5. Clustering
---

In [ ]:
from sklearn.cluster import KMeans

# Seleccionamos k
k_solucion = 5

# Instanciar
cluster = KMeans(n_clusters=k_solucion, n_init=10)

# Entrenar
cluster.fit_transform(df)

# Calcular segmento
df['segmento'] = cluster.predict(df)

# 7. Modelización
---

In [ ]:
from sklearn.linear_model import LogisticRegression

# Instanciamos
modelo = LogisticRegression()

"""
best_params = {'algoritmo': LogisticRegression(),
 'algoritmo__C': 1,
 'algoritmo__n_jobs': -1,
 'algoritmo__penalty': 'l1',
 'algoritmo__solver': 'saga'}
"""

pred = modelo.best_estimator_.predict_proba(val_x)[:,1]

"\n{'algoritmo': LogisticRegression(),\n 'algoritmo__C': 1,\n 'algoritmo__n_jobs': -1,\n 'algoritmo__penalty': 'l1',\n 'algoritmo__solver': 'saga'}\n"